In [3]:
import os

folder = "../data/raw/"

files = os.listdir(folder)

for file in files:
    print(file)

.gitkeep
extracted_csv
Pass-Fail Data.csv
Titanic Dataset.csv


In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/Pass-Fail Data.csv")

In [4]:
df.shape

(100, 6)

In [6]:
df.columns.tolist()

['student_id',
 'attendance_pct',
 'homework_pct',
 'midterm_score',
 'study_hours_per_week',
 'pass']

In [17]:
df.head()

,student_id,attendance_pct,homework_pct,midterm_score,study_hours_per_week,pass
0,1,95,92,88,12,1
1,2,88,85,79,10,1
2,3,60,55,58,4,0
3,4,72,70,65,6,1
4,5,40,45,50,3,0


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   student_id            100 non-null    int64
 1   attendance_pct        100 non-null    int64
 2   homework_pct          100 non-null    int64
 3   midterm_score         100 non-null    int64
 4   study_hours_per_week  100 non-null    int64
 5   pass                  100 non-null    int64
dtypes: int64(6)
memory usage: 4.8 KB


In [9]:
df.isnull().sum()

student_id              0
attendance_pct          0
homework_pct            0
midterm_score           0
study_hours_per_week    0
pass                    0
dtype: int64

In [10]:
df['pass'].value_counts()

pass
1    60
0    40
Name: count, dtype: int64

In [12]:
df['pass'].value_counts(normalize=True) * 100

pass
1    60.0
0    40.0
Name: proportion, dtype: float64

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df['student_id'].duplicated().sum()

np.int64(0)

In [15]:
df.describe()

,student_id,attendance_pct,homework_pct,midterm_score,study_hours_per_week,pass
count,100.000000,100.000000,100.00000,100.000000,100.00000,100.000000
mean,50.500000,69.520000,69.03000,68.780000,7.28000,0.600000
std,29.011492,17.651783,17.01304,14.717254,3.62115,0.492366
min,1.000000,30.000000,35.00000,42.000000,2.00000,0.000000
25%,25.750000,55.000000,55.00000,56.000000,4.00000,0.000000
50%,50.500000,72.500000,70.00000,68.000000,7.00000,1.000000
75%,75.250000,85.000000,85.00000,82.000000,10.00000,1.000000
max,100.000000,95.000000,96.00000,97.000000,15.00000,1.000000


In [16]:
df.groupby('pass').mean(numeric_only=True)

,student_id,attendance_pct,homework_pct,midterm_score,study_hours_per_week
pass,,,,,
0,51.800000,50.725,51.00,53.725000,3.700000
1,49.633333,82.050,81.05,78.816667,9.666667


In [19]:
df.groupby('pass')[[
    'attendance_pct',
    'homework_pct',
    'midterm_score',
    'study_hours_per_week'
]].agg(['min', 'max', 'mean'])

attendance_pct             homework_pct            midterm_score      \
                min max    mean          min max   mean           min max   
pass                                                                        
0                30  66  50.725           35  63  51.00            42  62   
1                67  95  82.050           65  96  81.05            60  97   

                study_hours_per_week                
           mean                  min max      mean  
pass                                                
0     53.725000                    2   5  3.700000  
1     78.816667                    6  15  9.666667

In [20]:
df.sort_values('attendance_pct')[[
    'attendance_pct',
    'homework_pct',
    'midterm_score',
    'study_hours_per_week',
    'pass'
]].head(20)

,attendance_pct,homework_pct,midterm_score,study_hours_per_week,pass
10,30,35,42,2,0
40,33,38,45,2,0
70,34,39,44,2,0
22,35,40,48,2,0
52,36,41,47,2,0
82,38,42,46,2,0
4,40,45,50,3,0
94,41,43,48,3,0
34,42,44,49,3,0
64,43,45,50,3,0


In [21]:
df.sort_values('attendance_pct')[[
    'attendance_pct',
    'homework_pct',
    'midterm_score',
    'study_hours_per_week',
    'pass'
]].tail(20)

,attendance_pct,homework_pct,midterm_score,study_hours_per_week,pass
87,87,85,83,10,1
65,88,91,93,14,1
18,88,90,86,12,1
1,88,85,79,10,1
78,88,90,87,12,1
54,89,88,86,11,1
95,89,92,94,14,1
38,89,87,83,11,1
8,90,88,84,11,1
24,90,88,85,11,1


In [22]:
def gini_impurity(y):
    total = len(y)

    pass_count = sum(y)
    fail_count = total - pass_count

    p_pass = pass_count / total
    p_fail = fail_count / total

    gini = 1 - (p_pass ** 2 + p_fail ** 2)

    return gini

In [23]:
gini_impurity(df["pass"])

0.48

In [24]:
def weighted_gini(y_left, y_right):
    total = len(y_left) + len(y_right)

    left_weight = len(y_left) / total
    right_weight = len(y_right) / total

    gini_left = gini_impurity(y_left)
    gini_right = gini_impurity(y_right)

    weighted = (
        left_weight * gini_left
        + right_weight * gini_right
    )

    return weighted

In [25]:
left = df[df["attendance_pct"] <= 66]["pass"]
right = df[df["attendance_pct"] > 66]["pass"]

weighted_gini(left, right)

0.0

In [26]:
features = [
    "attendance_pct",
    "homework_pct",
    "midterm_score",
    "study_hours_per_week"
]

In [27]:
def find_best_split(X, y):
    best_gini = float("inf")
    best_feature = None
    best_threshold = None

    for feature in X.columns:

        thresholds = sorted(X[feature].unique())

        for threshold in thresholds:

            left_mask = X[feature] <= threshold
            right_mask = X[feature] > threshold

            y_left = y[left_mask]
            y_right = y[right_mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            gini = weighted_gini(y_left, y_right)

            if gini < best_gini:
                best_gini = gini
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_gini

In [29]:
X = df[
    [
        "attendance_pct",
        "homework_pct",
        "midterm_score",
        "study_hours_per_week"
    ]
]

y = df["pass"]

In [30]:
best_feature, best_threshold, best_gini = find_best_split(X, y)

print("Best feature:", best_feature)
print("Best threshold:", best_threshold)
print("Best Gini:", best_gini)

Best feature: attendance_pct
Best threshold: 66
Best Gini: 0.0


In [31]:
import numpy as np

In [32]:
import numpy as np

# Set a random seed so that we get the same shuffle every time
np.random.seed(42)

# Create an array containing the row positions from 0 to 99
indices = np.arange(len(df))

# Randomly shuffle the row positions
np.random.shuffle(indices)

In [33]:
# Calculate the position where we will split the data
# 80% of the data will be used for training
split_index = int(0.8 * len(df))

# Select the first 80% of the shuffled indices for training
train_indices = indices[:split_index]

# Select the remaining 20% of the shuffled indices for testing
test_indices = indices[split_index:]

In [ ]:
# Create X_train, X_test, y_train and y_test
X_train = X.iloc[train_indices]
X_test = X.iloc[test_indices]

y_train = y.iloc[train_indices]
y_test = y.iloc[test_indices]

In [35]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (80, 4)
X_test: (20, 4)
y_train: (80,)
y_test: (20,)


In [ ]:
# find the best split using only training data
best_feature, best_threshold, best_gini = find_best_split(
    X_train,
    y_train
)

print("Best feature:", best_feature)
print("Best threshold:", best_threshold)
print("Best Gini:", best_gini)

Best feature: attendance_pct
Best threshold: 66
Best Gini: 0.0


In [38]:
class Node:
    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        value=None
    ):
        # Feature used to split the data
        self.feature = feature

        # Threshold used for the split
        self.threshold = threshold

        # Left child node
        self.left = left

        # Right child node
        self.right = right

        # Prediction stored in a leaf node
        self.value = value

In [39]:
def majority_class(y):
    # Count the number of Pass and Fail students
    pass_count = sum(y)
    fail_count = len(y) - pass_count

    # Return Pass if Pass is the majority
    if pass_count >= fail_count:
        return 1

    # Otherwise return Fail
    return 0

In [40]:
majority_class([1, 1, 1, 0, 0])

1

In [41]:
def build_tree(X, y, depth=0, max_depth=5):
    
    # Stop if the group is empty
    if len(y) == 0:
        return None

    # Stop if all students belong to the same class
    if len(set(y)) == 1:
        return Node(value=y.iloc[0])

    # Stop if we have reached the maximum tree depth
    if depth >= max_depth:
        return Node(value=majority_class(y))

    # Find the best feature and threshold for this group
    best_feature, best_threshold, best_gini = find_best_split(X, y)

    # Stop if no useful split was found
    if best_feature is None:
        return Node(value=majority_class(y))

    # Divide the data using the best split
    left_mask = X[best_feature] <= best_threshold
    right_mask = X[best_feature] > best_threshold

    # Create the left and right datasets
    X_left = X[left_mask]
    X_right = X[right_mask]

    # Create the left and right target groups
    y_left = y[left_mask]
    y_right = y[right_mask]

    # Recursively build the left side of the tree
    left_node = build_tree(
        X_left,
        y_left,
        depth + 1,
        max_depth
    )

    # Recursively build the right side of the tree
    right_node = build_tree(
        X_right,
        y_right,
        depth + 1,
        max_depth
    )

    # Return a decision node containing the split and child nodes
    return Node(
        feature=best_feature,
        threshold=best_threshold,
        left=left_node,
        right=right_node
    )

In [42]:
# Build the Decision Tree using only the training data
tree = build_tree(
    X_train,
    y_train,
    max_depth=5
)

In [43]:
# Check the type of object created by the tree-building function
print(type(tree))

<class '__main__.Node'>


In [44]:
def print_tree(node, depth=0):
    # Stop if there is no node
    if node is None:
        return

    # Create indentation based on the depth of the tree
    indent = "    " * depth

    # If the node is a leaf, display its prediction
    if node.value is not None:
        class_name = "Pass" if node.value == 1 else "Fail"
        print(f"{indent}Leaf: {class_name}")
        return

    # Display the decision rule at this node
    print(f"{indent}{node.feature} <= {node.threshold}")

    # Display the branch where the condition is True
    print(f"{indent}├── True:")
    print_tree(node.left, depth + 1)

    # Display the branch where the condition is False
    print(f"{indent}└── False:")
    print_tree(node.right, depth + 1)

In [45]:
# Display the structure learned by the Decision Tree
print_tree(tree)

attendance_pct <= 66
├── True:
    Leaf: Fail
└── False:
    Leaf: Pass


In [51]:
def predict_one(node, row):
    # If we reach a leaf node, return the prediction stored in that leaf
    if node.value is not None:
        return node.value

    # Follow the left branch if the student's value is
    # less than or equal to the threshold
    if row[node.feature] <= node.threshold:
        return predict_one(node.left, row)

    # Otherwise, follow the right branch
    return predict_one(node.right, row)

In [54]:
def predict(tree, X):
    # Create an empty list to store the predictions
    predictions = []

    # Go through each student in the dataset
    for _, row in X.iterrows():

        # Predict whether the student will Pass or Fail
        prediction = predict_one(tree, row)

        # Add the prediction to the list
        predictions.append(prediction)

    # Convert the predictions into a NumPy array
    return np.array(predictions)

In [55]:
# Use the trained Decision Tree to predict the test data
y_pred = predict(tree, X_test)

# Display the predictions
print(y_pred)

[1 1 0 0 1 0 1 0 1 1 1 1 0 0 0 1 1 1 0 1]


In [56]:
# Display the actual results and the model's predictions side by side
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

# Display the comparison
print(comparison)

    Actual  Predicted
0        1          1
1        1          1
2        0          0
3        0          0
4        1          1
5        0          0
6        1          1
7        0          0
8        1          1
9        1          1
10       1          1
11       1          1
12       0          0
13       0          0
14       0          0
15       1          1
16       1          1
17       1          1
18       0          0
19       1          1


In [57]:
# Compare the actual values with the predicted values
correct_predictions = sum(y_test.values == y_pred)

# Display the number of correct predictions
print("Correct predictions:", correct_predictions)

# Display the total number of test students
print("Total test students:", len(y_test))

Correct predictions: 20
Total test students: 20


In [58]:
# Count how many predictions match the actual results
correct_predictions = sum(y_test.values == y_pred)

# Calculate the total number of test observations
total_predictions = len(y_test)

# Calculate accuracy as the proportion of correct predictions
accuracy = correct_predictions / total_predictions

# Convert accuracy to a percentage
accuracy_percentage = accuracy * 100

# Display the results
print("Correct predictions:", correct_predictions)
print("Total predictions:", total_predictions)
print("Accuracy:", accuracy_percentage, "%")

Correct predictions: 20
Total predictions: 20
Accuracy: 100.0 %


In [59]:
# Count True Positives, True Negatives, False Positives, and False Negatives
TP = 0
TN = 0
FP = 0
FN = 0

# Compare each actual value with its predicted value
for actual, predicted in zip(y_test, y_pred):

    # Actual Pass and predicted Pass
    if actual == 1 and predicted == 1:
        TP += 1

    # Actual Fail and predicted Fail
    elif actual == 0 and predicted == 0:
        TN += 1

    # Actual Fail but predicted Pass
    elif actual == 0 and predicted == 1:
        FP += 1

    # Actual Pass but predicted Fail
    elif actual == 1 and predicted == 0:
        FN += 1

# Display the results
print("True Positives:", TP)
print("True Negatives:", TN)
print("False Positives:", FP)
print("False Negatives:", FN)

True Positives: 12
True Negatives: 8
False Positives: 0
False Negatives: 0


In [60]:
# Calculate precision
# Precision measures how many predicted Pass students actually passed
precision = TP / (TP + FP)

# Display precision as a percentage
print("Precision:", precision * 100, "%")

Precision: 100.0 %


In [61]:
# Calculate recall
# Recall measures how many actual Pass students were correctly identified
recall = TP / (TP + FN)

# Display recall as a percentage
print("Recall:", recall * 100, "%")

Recall: 100.0 %


In [62]:
# Calculate the F1 score using precision and recall
f1_score = 2 * (precision * recall) / (precision + recall)

# Display the F1 score as a percentage
print("F1 Score:", f1_score * 100, "%")

F1 Score: 100.0 %


In [63]:
# Store the different maximum depths we want to test
depths = [1, 2, 3, 4, 5]

# Store the accuracy for each tree
depth_accuracies = []

# Test each maximum depth
for depth in depths:

    # Build a new Decision Tree using the selected maximum depth
    test_tree = build_tree(
        X_train,
        y_train,
        max_depth=depth
    )

    # Use the tree to make predictions on the test data
    test_predictions = predict(
        test_tree,
        X_test
    )

    # Count the number of correct predictions
    correct = sum(y_test.values == test_predictions)

    # Calculate the accuracy
    accuracy = correct / len(y_test)

    # Store the accuracy
    depth_accuracies.append(accuracy)

# Display the results
for depth, accuracy in zip(depths, depth_accuracies):
    print(
        f"Max depth {depth}: "
        f"{accuracy * 100:.1f}% accuracy"
    )

Max depth 1: 100.0% accuracy
Max depth 2: 100.0% accuracy
Max depth 3: 100.0% accuracy
Max depth 4: 100.0% accuracy
Max depth 5: 100.0% accuracy


In [66]:
# Build the final Decision Tree using the simplest effective depth
final_tree = build_tree(
    X_train,
    y_train,
    max_depth=1
)

# Display the final tree structure
print_tree(final_tree)

attendance_pct <= 66
├── True:
    Leaf: Fail
└── False:
    Leaf: Pass


In [67]:
# Select the first student from the test dataset
student = X_test.iloc[0]

# Display the student's feature values
print(student)

attendance_pct          74
homework_pct            72
midterm_score           69
study_hours_per_week     7
Name: 63, dtype: int64


In [68]:
# Use our Decision Tree to predict this student's result
prediction = predict_one(final_tree, student)

# Convert the numeric prediction into a readable label
result = "Pass" if prediction == 1 else "Fail"

# Display the prediction
print("Predicted result:", result)

Predicted result: Pass


In [69]:
# Get the actual result of the selected test student
actual = y_test.iloc[0]

# Convert the numeric actual value into a readable label
actual_result = "Pass" if actual == 1 else "Fail"

# Display both the actual result and the model prediction
print("Actual result:", actual_result)
print("Predicted result:", result)

Actual result: Pass
Predicted result: Pass


In [70]:
# Calculate the main evaluation metrics for the final Decision Tree

# Accuracy measures the overall percentage of correct predictions
accuracy = (TP + TN) / (TP + TN + FP + FN)

# Precision measures how many predicted Pass students actually passed
precision = TP / (TP + FP)

# Recall measures how many actual Pass students were correctly identified
recall = TP / (TP + FN)

# F1 combines precision and recall into one score
f1 = 2 * (precision * recall) / (precision + recall)

# Display the final evaluation results
print("Final Decision Tree Evaluation")
print("--------------------------------")
print(f"Accuracy:  {accuracy * 100:.1f}%")
print(f"Precision: {precision * 100:.1f}%")
print(f"Recall:    {recall * 100:.1f}%")
print(f"F1 Score:  {f1 * 100:.1f}%")

Final Decision Tree Evaluation
--------------------------------
Accuracy:  100.0%
Precision: 100.0%
Recall:    100.0%
F1 Score:  100.0%
